# Import and Register

Loads models from the import volume and registers them in the target catalog. Idempotent (skips already-imported versions). Restores Champion/Challenger/Shadow aliases. Model names in target match source (no suffix).

## Configuration

Read widget parameters: source/target catalogs, schema, volume catalog, and import volume name.

In [ ]:
dbutils.widgets.text("source_catalog", "", "Source catalog")
dbutils.widgets.text("target_catalog", "", "Target catalog")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("import_volume", "model_imports", "Import volume name")
dbutils.widgets.text("volume_catalog", "", "Catalog holding import volume (empty = target_catalog)")

TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip()
TARGET_SCHEMA = dbutils.widgets.get("schema").strip()
IMPORT_VOLUME = dbutils.widgets.get("import_volume").strip()
VOLUME_CATALOG = dbutils.widgets.get("volume_catalog").strip() or TARGET_CATALOG
VOLUME_ROOT = f"/Volumes/{VOLUME_CATALOG}/{TARGET_SCHEMA}/{IMPORT_VOLUME}"

if not TARGET_CATALOG:
    raise ValueError("Set target_catalog")
try:
    user_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
except Exception:
    try:
        user_name = spark.conf.get("spark.databricks.userContext.userName")
    except Exception:
        user_name = "unknown"
print(f"Volume: {VOLUME_ROOT}")

## MLflow client and helper functions

Set up the MLflow UC registry client and define helpers for extracting model signatures from metadata, artifacts, or by inference.

In [ ]:
import mlflow, json, os, time
import numpy as np, pandas as pd
from datetime import datetime
from mlflow import MlflowClient
from mlflow.models import infer_signature, ModelSignature
from mlflow.types.schema import Schema

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

def get_signature_from_metadata(meta):
    sig_dict = meta.get("signature") if isinstance(meta, dict) else None
    if not sig_dict or not isinstance(sig_dict, dict): return None
    inp, out = sig_dict.get("inputs"), sig_dict.get("outputs")
    if not inp: return None
    try:
        return ModelSignature(inputs=Schema.from_dict(inp), outputs=Schema.from_dict(out) if out else None)
    except Exception: return None

def get_signature_from_artifact(artifact_dir):
    if not artifact_dir or not os.path.isdir(artifact_dir): return None
    try:
        mi = mlflow.models.get_model_info(f"file://{os.path.abspath(artifact_dir)}")
        sig = getattr(mi, "signature", None)
        if sig and sig.inputs:
            return ModelSignature(inputs=Schema.from_dict(sig.inputs.to_dict()), outputs=Schema.from_dict(sig.outputs.to_dict()) if sig.outputs else None)
    except Exception: pass
    return None

def infer_signature_from_model(model, params=None, max_features=128):
    rng = np.random.default_rng(42)
    n = getattr(model, "n_features_in_", None)
    if n is None and params:
        try: n = int(params.get("n_features", 0)) or None
        except (TypeError, ValueError): n = None
        if n is None:
            fn = params.get("feature_names", "") or params.get("feature_names_str", "")
            if isinstance(fn, str) and fn: n = len([x.strip() for x in fn.split(",") if x.strip()])
            elif isinstance(fn, (list, tuple)): n = len(fn)
    if n and n > 0:
        for use_df in (True, False):
            try:
                dummy = pd.DataFrame(rng.normal(0, 1, (2, n)), columns=[f"f{i}" for i in range(n)]) if use_df else np.asarray(rng.normal(0, 1, (2, n)), dtype=np.float64)
                return infer_signature(dummy, model.predict(dummy))
            except Exception: continue
    for n in range(1, max_features + 1):
        try:
            dummy = pd.DataFrame(rng.normal(0, 1, (2, n)), columns=[f"f{i}" for i in range(n)])
            return infer_signature(dummy, model.predict(dummy))
        except Exception: continue
    return None
print(f"MLflow {mlflow.__version__}")

## Read import manifest

Loads the global manifest from the import volume to get the list of models and version counts to import.

In [ ]:
manifest_path = f"{VOLUME_ROOT}/manifest.json"
if not os.path.isfile(manifest_path):
    raise FileNotFoundError(f"Manifest not found: {manifest_path}. Run export & transfer first.")
with open(manifest_path) as f:
    global_manifest = json.load(f)
models_to_import = global_manifest["models"]
print(f"Models to import: {len(models_to_import)}")
for m in models_to_import:
    print(f"  {m['source']} -> {m['target']} ({m['versions']} versions)")

## Import models and restore aliases

For each model in the manifest: loads the sklearn artifact, logs it to a new MLflow run with preserved metrics/params/tags, registers it in the target catalog, and restores Champion/Challenger/Shadow aliases. Idempotent: skips versions that were already imported (matched by source run ID or source version tag).

In [ ]:
all_model_results = []
for model_entry in models_to_import:
    source_model = model_entry["source"]
    target_model = model_entry["target"]
    model_dir, model_root = model_entry["dir"], f"{VOLUME_ROOT}/{model_entry['dir']}"
    model_short_name = source_model.split(".")[-1]
    experiment_name = f"/Users/{user_name}/migration_{model_short_name}"
    exp = client.get_experiment_by_name(experiment_name)
    exp_id = exp.experiment_id if exp else client.create_experiment(experiment_name)
    mlflow.set_experiment(experiment_name)
    if not os.path.isdir(model_root):
        raise FileNotFoundError(f"Model directory not found: {model_root}")
    version_dirs = sorted([d for d in os.listdir(model_root) if d.startswith("v") and os.path.isdir(f"{model_root}/{d}")], key=lambda x: int(x[1:]))
    existing_tvs = list(client.search_model_versions(filter_string=f"name='{target_model}'"))
    existing_src_run_ids = set()
    existing_src_keys = set()
    for tv in existing_tvs:
        try:
            tr = client.get_run(tv.run_id)
            tt = dict(tr.data.tags)
            if tt.get("migration.source_run_id"): existing_src_run_ids.add(tt["migration.source_run_id"])
            sm, sv = tt.get("migration.source_model"), tt.get("migration.source_version")
            if sm and sv: existing_src_keys.add((sm, sv))
        except Exception: pass
    results = []
    for vdir_name in version_dirs:
        vnum, vpath = vdir_name[1:], f"{model_root}/{vdir_name}"
        meta_file, model_dir_path = f"{vpath}/metadata.json", f"{vpath}/model"
        if not os.path.exists(meta_file):
            results.append({"v": vnum, "ok": False}); continue
        with open(meta_file) as f: meta = json.load(f)
        metrics, params, tags = meta.get("metrics", {}), meta.get("params", {}), meta.get("tags", {})
        src_run = meta.get("source_run_id", "")
        if src_run in existing_src_run_ids or (source_model, vnum) in existing_src_keys:
            results.append({"v": vnum, "ok": True, "skipped": True}); continue
        if not os.path.isdir(model_dir_path) or not os.path.isfile(f"{model_dir_path}/MLmodel"):
            if os.path.isfile(f"{vpath}/MLmodel"): model_dir_path = vpath
            else: results.append({"v": vnum, "ok": False}); continue
        try:
            model = mlflow.sklearn.load_model(model_dir_path)
            signature = get_signature_from_metadata(meta) or get_signature_from_artifact(model_dir_path) or infer_signature_from_model(model, params=params)
            if signature is None:
                raise ValueError(f"UC requires a signature; none for {source_model} v{vnum}")
            input_example = None
            if params.get("feature_names"):
                fn = [f.strip() for f in params["feature_names"].split(",") if f.strip()]
                input_example = pd.DataFrame(np.random.default_rng(42).normal(0, 1, (1, len(fn))), columns=fn)
            run_name = f"migrated_v{vnum}_{model_short_name}"
            with mlflow.start_run(experiment_id=exp_id, run_name=run_name) as run:
                for k, v in params.items():
                    try: mlflow.log_param(k, v)
                    except Exception: pass
                for k, v in metrics.items():
                    try: mlflow.log_metric(k, float(v))
                    except Exception: pass
                for k, v in tags.items():
                    if k.startswith("mlflow."): continue
                    try: mlflow.set_tag(k, v)
                    except Exception: pass
                mlflow.set_tag("migration.source_model", source_model)
                mlflow.set_tag("migration.target_model", target_model)
                mlflow.set_tag("migration.source_version", vnum)
                mlflow.set_tag("migration.source_run_id", src_run)
                mlflow.set_tag("migration.type", "volume_deep_clone")
                _pre = set(v.version for v in client.search_model_versions(f"name='{target_model}'"))
                for attempt in range(3):
                    try:
                        mlflow.sklearn.log_model(sk_model=model, name="model", signature=signature, input_example=input_example, registered_model_name=target_model)
                        break
                    except Exception as e:
                        try:
                            _post = set(v.version for v in client.search_model_versions(f"name='{target_model}'"))
                            if _post - _pre: break
                        except Exception: pass
                        if attempt < 2 and ("500" in str(e) or "Retry" in str(e) or "too many" in str(e)):
                            time.sleep(10 * (2 ** attempt))
                        else: raise
            results.append({"v": vnum, "ok": True})
            existing_src_run_ids.add(src_run)
            existing_src_keys.add((source_model, vnum))
        except Exception as e:
            print(f"  v{vnum} FAILED: {e}")
            results.append({"v": vnum, "ok": False})
        time.sleep(1)
    alias_map = {}
    for vd in version_dirs:
        mf = f"{model_root}/{vd}/metadata.json"
        if os.path.exists(mf):
            with open(mf) as f:
                for a in json.load(f).get("aliases", []): alias_map[a] = vd[1:]
    if alias_map:
        all_tv = client.search_model_versions(filter_string=f"name='{target_model}'")
        src_to_tgt = {}
        for tv in all_tv:
            try:
                sv = client.get_run(tv.run_id).data.tags.get("migration.source_version")
                if sv and (sv not in src_to_tgt or int(tv.version) > int(src_to_tgt[sv])): src_to_tgt[sv] = tv.version
            except Exception: pass
        for alias, src_ver in alias_map.items():
            tgt_ver = src_to_tgt.get(src_ver)
            if tgt_ver:
                try: client.set_registered_model_alias(target_model, alias, tgt_ver); print(f"  {alias} -> v{tgt_ver}")
                except Exception as e: print(f"  {alias} FAILED: {e}")
    all_model_results.append({"source": source_model, "target": target_model, "ok": sum(1 for r in results if r["ok"]), "fail": sum(1 for r in results if not r["ok"]), "experiment": experiment_name})

print("="*60)
for r in all_model_results:
    print(f"  {r['source']} -> {r['target']}  ok={r['ok']}  fail={r['fail']}")
print("="*60)